# Cálculo y Publicación Batch de KPIs de Negocio

## 1. Inicializar la SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("criptoanalytics-s11-bi-ml")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", "3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## 2. Cargar el Parquet de Observabilidad de S09

In [2]:
RUTA_OBS_S9 = "/home/jovyan/work/artifacts/btc_trades_observabilidad"

df_obs = spark.read.parquet(RUTA_OBS_S9)
print(f"Filas de observabilidad cargadas: {df_obs.count():,}")


Filas de observabilidad cargadas: 388


## 3. Calcular KPIs de Negocio y Financieros

Se calculan métricas clave para el mercado de criptoactivos (ej. volumen total operado en USDT, precio promedio y calidad/validez del flujo)

In [3]:
kpis_negocio = df_obs.agg(
    F.count("*").alias("total_trades"),
    F.sum("quoteQuantity").alias("volumen_total_usdt"),
    F.avg("price").alias("precio_promedio_btc"),
    F.avg("latencyMs").alias("latencia_promedio_ms")
).first()

print(f"Total Trades: {kpis_negocio['total_trades']:,}")
print(f"Volumen Total (USDT): {kpis_negocio['volumen_total_usdt']:,.2f}")
print(f"Precio Promedio (USDT): {kpis_negocio['precio_promedio_btc']:,.2f}")
print(f"Latencia Promedio (ms): {kpis_negocio['latencia_promedio_ms']:.2f}")


Total Trades: 388
Volumen Total (USDT): 179,414.51
Precio Promedio (USDT): 81,291.20
Latencia Promedio (ms): 2603.77


## 4. Publicar el KPI Batch en Pushgateway

Se usa una petición HTTP PUT para enviar el estado completo de los KPIs una sola vez

In [4]:
import urllib.request

metricas_batch = {
    "cripto_bi_trades_total": kpis_negocio["total_trades"],
    "cripto_bi_volumen_usdt": kpis_negocio["volumen_total_usdt"],
    "cripto_bi_precio_promedio": kpis_negocio["precio_promedio_btc"],
    "cripto_bi_latencia_ms": kpis_negocio["latencia_promedio_ms"]
}

lineas = [f"# TYPE {k} gauge\n{k} {v}" for k, v in metricas_batch.items()]
cuerpo = "\n".join(lineas) + "\n"

url_push = "http://pushgateway:9091/metrics/job/cripto_bi_batch/instance/s11_batch_run"
req = urllib.request.Request(url_push, data=cuerpo.encode("utf-8"), method="PUT")

with urllib.request.urlopen(req) as resp:
    print("Pushgateway (Batch) respondió con estado:", resp.status)


Pushgateway (Batch) respondió con estado: 200


# Inferencia en Streaming y Publicación en Vivo

## 1. Cargar el Modelo Entrenado de S10

In [5]:
from pyspark.ml import PipelineModel

RUTA_MODELO_S10 = "/home/jovyan/work/artifacts/models/modelo_btc_lr"

modelo_cargado = PipelineModel.load(RUTA_MODELO_S10)
print("Modelo de regresión cargado exitosamente desde:", RUTA_MODELO_S10)


Modelo de regresión cargado exitosamente desde: /home/jovyan/work/artifacts/models/modelo_btc_lr


## 2. Configurar el Stream desde Kafka

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, BooleanType

esquema_crypto = (
    StructType()
    .add("tipoEvento", StringType())
    .add("tradeId", LongType())
    .add("symbol", StringType())
    .add("price", DoubleType())
    .add("quantity", DoubleType())
    .add("quoteQuantity", DoubleType())
    .add("isBuyerMaker", BooleanType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

stream_crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "binance-btc-trades")
    .option("startingOffsets", "latest")
    .load()
)

eventos_stream = (
    stream_crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(F.from_json(F.col("json_str"), esquema_crypto).alias("e"))
    .select("e.*")
)

# Crear la columna numérica buyer_maker_num exigida por el modelo guardado de S10
eventos_preparados = eventos_stream.withColumn(
    "buyer_maker_num", 
    F.when(F.col("isBuyerMaker") == True, 1.0).otherwise(0.0)
)

# Aplicar el modelo transformador cargado con las columnas correctas listas
predicciones_stream = modelo_cargado.transform(eventos_preparados)

## 3. Publicación Micro-lote por Micro-lote en Pushgateway (foreachBatch)

In [13]:
PATH_PARQUET_PRED = "/home/jovyan/work/artifacts/predicciones_stream_crypto"
CHECKPOINT_PRED = "/home/jovyan/work/artifacts/chk_predicciones_stream_crypto"

PUSHGATEWAY_STREAM_BASE = "http://pushgateway:9091/metrics/job/cripto_ml_streaming/instance"

def publicar_en_vivo(symbol, precio_real, prediccion):
    cuerpo = (
        "# TYPE cripto_ml_precio_real gauge\n"
        f"cripto_ml_precio_real {precio_real}\n"
        "# TYPE cripto_ml_prediccion_siguiente gauge\n"
        f"cripto_ml_prediccion_siguiente {prediccion}\n"
    )
    url = f"{PUSHGATEWAY_STREAM_BASE}/{symbol}"
    req = urllib.request.Request(url, data=cuerpo.encode("utf-8"), method="PUT")
    urllib.request.urlopen(req).close()

def procesar_microlote(df_lote, batch_id):
    conteo = df_lote.count()
    if conteo == 0:
        return
    
    # Persistir en Parquet en modo append
    df_lote.select(
        F.lit(batch_id).alias("batchId"), "symbol", "price", 
        F.col("prediction").alias("prediccionSiguientePrecio"),
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora")
    ).write.mode("append").parquet(PATH_PARQUET_PRED)

    # Enviar al Pushgateway en tiempo real
    for row in df_lote.select("symbol", "price", "prediction").collect():
        publicar_en_vivo(row["symbol"], row["price"], row["prediction"])
        
    print(f"Micro-lote {batch_id}: {conteo} predicciones procesadas y publicadas en vivo.")

# Iniciar la consulta en streaming
query_streaming = (
    predicciones_stream.writeStream
    .foreachBatch(procesar_microlote)
    .option("checkpointLocation", CHECKPOINT_PRED)
    .trigger(processingTime="5 seconds")
    .start()
)


Micro-lote 53: 62577 predicciones procesadas y publicadas en vivo.
Micro-lote 54: 330 predicciones procesadas y publicadas en vivo.
Micro-lote 55: 2 predicciones procesadas y publicadas en vivo.
Micro-lote 56: 8 predicciones procesadas y publicadas en vivo.
Micro-lote 57: 135 predicciones procesadas y publicadas en vivo.
Micro-lote 58: 30 predicciones procesadas y publicadas en vivo.
Micro-lote 59: 10 predicciones procesadas y publicadas en vivo.
Micro-lote 60: 10 predicciones procesadas y publicadas en vivo.
Micro-lote 61: 364 predicciones procesadas y publicadas en vivo.
Micro-lote 62: 271 predicciones procesadas y publicadas en vivo.
Micro-lote 63: 8 predicciones procesadas y publicadas en vivo.
Micro-lote 64: 238 predicciones procesadas y publicadas en vivo.
Micro-lote 65: 112 predicciones procesadas y publicadas en vivo.
Micro-lote 66: 355 predicciones procesadas y publicadas en vivo.
Micro-lote 67: 656 predicciones procesadas y publicadas en vivo.
Micro-lote 68: 7 predicciones pr

# Confirmación en Vivo con PromQL

In [11]:
import json
import time

def consultar_prometheus(metrica):
    url = f"http://prometheus:9090/api/v1/query?query={metrica}"
    res = json.load(urllib.request.urlopen(url))
    resultado = res["data"]["result"]
    return resultado[0]["value"][1] if resultado else None

val_1 = consultar_prometheus("cripto_ml_precio_real")
print("Primera lectura en Prometheus:", val_1)

print("Esperando 15 segundos para capturar nuevos trades...")
time.sleep(15)

val_2 = consultar_prometheus("cripto_ml_precio_real")
print("Segunda lectura (15s después):", val_2)

if val_1 != val_2:
    print("¡Confirmado! La métrica en vivo cambió correctamente entre consultas.")
else:
    print("Aviso: La métrica se mantuvo igual o el productor no emitió datos nuevos en ese lapso.")


Primera lectura en Prometheus: 81296.8
Esperando 15 segundos para capturar nuevos trades...
Micro-lote 36: 182 predicciones procesadas y publicadas en vivo.
Micro-lote 37: 30 predicciones procesadas y publicadas en vivo.
Micro-lote 38: 225 predicciones procesadas y publicadas en vivo.
Segunda lectura (15s después): 81303.99
¡Confirmado! La métrica en vivo cambió correctamente entre consultas.
Micro-lote 39: 13 predicciones procesadas y publicadas en vivo.
Micro-lote 40: 415 predicciones procesadas y publicadas en vivo.
Micro-lote 41: 110 predicciones procesadas y publicadas en vivo.
Micro-lote 42: 144 predicciones procesadas y publicadas en vivo.
Micro-lote 43: 9 predicciones procesadas y publicadas en vivo.
Micro-lote 44: 166 predicciones procesadas y publicadas en vivo.
Micro-lote 45: 200 predicciones procesadas y publicadas en vivo.
Micro-lote 46: 8 predicciones procesadas y publicadas en vivo.
Micro-lote 47: 5 predicciones procesadas y publicadas en vivo.
Micro-lote 48: 54 predicci

In [14]:
query_streaming.stop()
print("Consulta detenida.")

Consulta detenida.
